# Baseline Models

In [1]:
from transformers import AutoTokenizer, AutoConfig
from transformers import AutoModelForSequenceClassification
from transformers import pipeline

import polars as pl
import numpy as np
import time

from pyprojroot import here
from scipy.special import softmax
from ast import literal_eval

/Users/paulterrasi/Documents/Political-Signaling-Sentiment-Analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0406 19:17:39.211000 95635 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []
    for t in text.split():
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

# Setup model and tokenizer
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# Load train and test data
train = pl.read_parquet(here("data/train.parquet"))
test = pl.read_parquet(here("data/test.parquet"))

In [4]:
# Test inference logic

tst = train.head(10).collect().row(2, named=True)

print(tst)

text = preprocess(tst["content"])
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)
scores = softmax(output[0][0].detach().numpy())
{label: score for label, score in zip(config.id2label.values(), scores)}

{'index': 3, 'id': 'c0345dl', 'subreddit': 'movies', 'username': 'ataraxis', 'username_score': 0.0, 'content': "'1. There Will Be Blood 9.1 (1,151)\\n2. No Country For Old Men 8.9 (21,431)\\n3. Juno 8.6 (3,352)\\n4. In the Shadow of the Moon 8.6 (705)\\n5. Sicko 8.5 (20,557)'", 'label': 'neutral'}


{'negative': np.float32(0.12865186),
 'neutral': np.float32(0.827388),
 'positive': np.float32(0.04396018)}

In [5]:
def inference_single(content: str) -> dict[str, np.float32]:
    text = preprocess(literal_eval(content))
    encoded_input = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    output = model(**encoded_input)
    scores = softmax(output[0][0].detach().numpy())
    return {label: float(score) for label, score in zip(config.id2label.values(), scores)}

In [6]:
inference_single(test.head(10).collect().row(2, named=True)["content"])

{'negative': 0.8711820244789124,
 'neutral': 0.11756552755832672,
 'positive': 0.0112524488940835}

In [ ]:
tst = train.head(500)

tst.with_columns(
    pl.col("content")
    .map_elements(inference_single, return_dtype=pl.Struct({'negative': pl.Float64, 'neutral': pl.Float64, 'positive': pl.Float64}))
    .alias("score")
).unnest("score")

index,id,subreddit,username,username_score,content,label,negative,neutral,positive
u32,str,str,str,f64,str,str,f64,f64,f64
0,"""c02zwul""","""movies""","""Mendokusai""",0.0,"""'The Borat one is a little ove…","""neutral""",0.29689,0.654079,0.049031
2,"""c0343cf""","""movies""","""bighippo""",0.0,"""'great site for streaming vide…","""neutral""",0.003129,0.035986,0.960885
3,"""c0345dl""","""movies""","""ataraxis""",0.0,"""'1. There Will Be Blood 9.1 (1…","""neutral""",0.142474,0.81452,0.043006
4,"""c034g4l""","""movies""","""qtoo""",0.0,"""""Don't forget Ryan Vs. Dorkman…","""neutral""",0.006704,0.793853,0.199443
6,"""c0365ub""","""movies""","""Mendokusai""",0.0,"""'Maybe 1 out of 5 are actually…","""neutral""",0.17886,0.702852,0.118288
…,…,…,…,…,…,…,…,…,…
635,"""c05ap4b""","""movies""","""beavershaw""",0.0,"""""I'm a bit surprised by Dead M…","""neutral""",0.858118,0.128611,0.013271
637,"""c05ap7r""","""movies""","""beavershaw""",0.0,"""""It's clearly a mistake.\n\nht…","""neutral""",0.869537,0.122642,0.007822
638,"""c05apov""","""movies""","""beavershaw""",0.0,"""'Am I the only one who actuall…","""neutral""",0.022113,0.094311,0.883576


: 

In [8]:
def inference_batch(batch: pl.Series) -> dict[str, np.float32]:
    times = [time.perf_counter()]
    
    text = batch.map_elements(literal_eval).map_elements(preprocess)
    times.append(time.perf_counter())
    print(times[-1] - times[-2])

    encoded_input = tokenizer(text.to_list(), return_tensors='pt', padding=True, truncation=True, max_length=512)
    times.append(time.perf_counter())
    print(times[-1] - times[-2])

    output = model(**encoded_input)
    times.append(time.perf_counter())
    print(times[-1] - times[-2])

    scores = output[0].detach().numpy()
    times.append(time.perf_counter())
    print(times[-1] - times[-2])

    return pl.Series([{label: float(score) for label, score in zip(config.id2label.values(), s)} for s in scores])

In [ ]:
tst = train.head(500)

tst = tst.with_columns(
    pl.col("content")
    .map_batches(inference_batch, return_dtype=pl.Struct({'negative': pl.Float64, 'neutral': pl.Float64, 'positive': pl.Float64}))
    .alias("score")
).unnest("score")

exp_sum = (pl.col("negative").exp() + pl.col("neutral").exp() + pl.col("positive").exp()).alias("exp_sum")

tst.with_columns([
    exp_sum.alias("exp_sum"),
    pl.col("negative").exp() / exp_sum,
    pl.col("neutral").exp() / exp_sum,
    pl.col("positive").exp() / exp_sum,
    pl.when(pl.col("negative") > pl.col("neutral")).then(
        pl.when(pl.col("negative") > pl.col("positive"))
        .then(pl.lit("negative"))
        .otherwise(pl.lit("positive"))
    ).otherwise(pl.when(pl.col("neutral") > pl.col("positive"))
        .then(pl.lit("neutral"))
        .otherwise(pl.lit("positive"))
    )
]).drop("exp_sum")

0.006899083004100248
0.18831345799844712
